<a href="https://colab.research.google.com/github/egxl/Turnitin_Similaritas_P3MD/blob/main/Turnitin_Similaritas_P3MD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🔍 Pemeriksa Similaritas Turnitin-Style Dokumen P3MD (Incremental & Web UI)
Notebook ini mereplikasi mekanisme resmi **Turnitin** dalam mendeteksi similaritas teks antardokumen dengan optimasi performa tinggi:
- **⚡ Pemrosesan Inkremental (SQLite Cache):** Dokumen yang sudah diproses tidak akan dihitung ulang! Saat ada dokumen baru ditambahkan, sistem hanya membandingkan dokumen baru tersebut terhadap database (menghemat 99% waktu komputasi).
- **🚀 Sinkronisasi Otomatis Google Drive Publik:** Seluruh peserta cukup mengunggah naskah tugas ke folder publik Google Drive yang disediakan (tersedia tombol tautan langsung di antarmuka Web UI). Sistem otomatis mengunduh dokumen baru secara cerdas dengan pagination Drive API (`pageSize=1000`), melewati file yang sudah ada, tanpa perlu konfigurasi folder lokal atau *drive mount*.
- **🌐 Tampilan Web UI Modern & Real-Time Progress Bar (Gradio):** Dilengkapi dashboard antarmuka web interaktif dengan bilah kemajuan (*real-time progress bar*), panduan alur kerja langsung di layar, rekap leaderboard per peserta, dan tombol unduh laporan resmi Excel 7-Sheet. Dapat diakses publik (`.gradio.live`) gratis selama 72 jam per sesi!
- **📊 Standar Turnitin:** K-Gram Shingling (6-8 kata), Word-Level Containment Index, filter Kutipan (*Quotes*), filter Daftar Pustaka (*Bibliography*), dan warna badge Turnitin (🔵 Blue, 🟢 Green, 🟡 Yellow, 🟠 Orange, 🔴 Red).


In [ ]:
# @title 1. Instalasi Library Pendukung
!pip install -q gradio python-docx pypdf openpyxl pandas google-api-python-client
print("✅ Library siap digunakan.")


In [ ]:
# @title 2. Core Turnitin Engine & Incremental SQLite Cache
import os
import re
import math
import json
import sqlite3
import pandas as pd
from itertools import combinations
from docx import Document
from pypdf import PdfReader

# Threshold Turnitin resmi
COMMANDER_THRESHOLD = 17.0
PASS_THRESHOLD = 15.0

def turnitin_badge(score):
    if score == 0: return "🔵 Blue (0%)"
    elif score < 25: return "🟢 Green (1-24%)"
    elif score < 50: return "🟡 Yellow (25-49%)"
    elif score < 75: return "🟠 Orange (50-74%)"
    else: return "🔴 Red (75-100%)"

def extract_raw_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    try:
        if ext == ".docx":
            doc = Document(file_path)
            text = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
        elif ext == ".pdf":
            reader = PdfReader(file_path)
            text = "\n".join([page.extract_text() or "" for page in reader.pages])
        elif ext == ".txt":
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
    except Exception as e:
        print(f"⚠️ Gagal mengekstrak {os.path.basename(file_path)}: {e}")
    return text

def apply_turnitin_exclusions(text, drop_quotes=True, drop_bib=True):
    """Menerapkan filter eksklusi Turnitin: kutipan dan daftar pustaka."""
    if drop_bib:
        bib_pattern = r"\n\s*(?:daftar\s+pustaka|references|bibliography|rujukan)\s*[:\n]"
        parts = re.split(bib_pattern, text, flags=re.IGNORECASE)
        if len(parts) > 1:
            text = parts[0]

    if drop_quotes:
        text = re.sub(r'"[^"]*"|[“”][^“”]*[“”]', ' ', text)

    return text

def tokenize_words(text):
    return re.findall(r"\b\w+\b", text.lower())

def build_kgram_map(words, k=6):
    """Membangun map dari k-gram ke daftar index posisi munculnya."""
    kgram_map = {}
    for i in range(len(words) - k + 1):
        gram = " ".join(words[i:i+k])
        if gram not in kgram_map:
            kgram_map[gram] = []
        kgram_map[gram].append(i)
    return kgram_map

def calculate_turnitin_similarity(doc_a_words, doc_b_words, map_a, map_b, k=6):
    """Kalkulasi similaritas Turnitin menggunakan precomputed k-gram map."""
    if len(doc_a_words) < k or len(doc_b_words) < k:
        return 0.0, 0.0, []

    common_grams = set(map_a.keys()) & set(map_b.keys())
    if not common_grams:
        return 0.0, 0.0, []

    matched_indices_a = set()
    for gram in common_grams:
        for start_idx in map_a[gram]:
            for offset in range(k):
                matched_indices_a.add(start_idx + offset)

    matched_indices_b = set()
    for gram in common_grams:
        for start_idx in map_b[gram]:
            for offset in range(k):
                matched_indices_b.add(start_idx + offset)

    score_a = (len(matched_indices_a) / len(doc_a_words) * 100) if doc_a_words else 0.0
    score_b = (len(matched_indices_b) / len(doc_b_words) * 100) if doc_b_words else 0.0

    passages = []
    if matched_indices_a:
        sorted_indices = sorted(matched_indices_a)
        current_passage = [doc_a_words[sorted_indices[0]]]
        for prev_idx, curr_idx in zip(sorted_indices[:-1], sorted_indices[1:]):
            if curr_idx == prev_idx + 1:
                current_passage.append(doc_a_words[curr_idx])
            else:
                passages.append(" ".join(current_passage))
                current_passage = [doc_a_words[curr_idx]]
        passages.append(" ".join(current_passage))

    return score_a, score_b, passages

class TurnitinDBCache:
    """Manajer SQLite Cache untuk pemrosesan inkremental Turnitin."""
    def __init__(self, db_path="similarity_cache.db"):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        conn = sqlite3.connect(self.db_path)
        try:
            cur = conn.cursor()
            cur.execute("""
                CREATE TABLE IF NOT EXISTS documents (
                    filename TEXT PRIMARY KEY,
                    file_mtime REAL,
                    file_size INTEGER,
                    word_count INTEGER,
                    words_json TEXT
                )
            """)
            cur.execute("""
                CREATE TABLE IF NOT EXISTS pairs (
                    doc_a TEXT,
                    doc_b TEXT,
                    k_val INTEGER,
                    max_score REAL,
                    score_a REAL,
                    score_b REAL,
                    matches INTEGER,
                    passages_json TEXT,
                    badge TEXT,
                    status TEXT,
                    PRIMARY KEY (doc_a, doc_b, k_val)
                )
            """)
            conn.commit()
        finally:
            conn.close()

    def sync_documents(self, file_paths, drop_quotes=True, drop_bib=True, progress_callback=None):
        """Memproses hanya dokumen yang baru atau berubah."""
        doc_database = {}
        new_or_updated = 0
        loaded_from_cache = 0
        total_files = len(file_paths)

        conn = sqlite3.connect(self.db_path)
        try:
            cur = conn.cursor()
            for idx, fp in enumerate(file_paths):
                name = os.path.basename(fp)
                if progress_callback:
                    progress_callback(idx + 1, total_files, name)

                mtime = os.path.getmtime(fp)
                size = os.path.getsize(fp)

                cur.execute("SELECT file_mtime, file_size, words_json FROM documents WHERE filename = ?", (name,))
                row = cur.fetchone()

                if row and row[0] == mtime and row[1] == size:
                    words = json.loads(row[2])
                    doc_database[name] = words
                    loaded_from_cache += 1
                else:
                    raw = extract_raw_text(fp)
                    filtered = apply_turnitin_exclusions(raw, drop_quotes=drop_quotes, drop_bib=drop_bib)
                    words = tokenize_words(filtered)
                    if len(words) > 0:
                        doc_database[name] = words
                        cur.execute("""
                            INSERT OR REPLACE INTO documents (filename, file_mtime, file_size, word_count, words_json)
                            VALUES (?, ?, ?, ?, ?)
                        """, (name, mtime, size, len(words), json.dumps(words)))
                        if row is not None:
                            # Dokumen yang sudah ada diperbarui: hapus data pasangan lama agar dihitung ulang otomatis
                            cur.execute("DELETE FROM pairs WHERE doc_a = ? OR doc_b = ?", (name, name))
                        new_or_updated += 1
            conn.commit()
        finally:
            conn.close()

        return doc_database, loaded_from_cache, new_or_updated

    def run_incremental_comparisons(self, doc_database, k_val=6, threshold=PASS_THRESHOLD, max_passages=5, progress_callback=None):
        """Menghitung hanya pasangan dokumen yang belum pernah dianalisis."""
        names = sorted(list(doc_database.keys()))
        if len(names) < 2:
            return 0, 0, 0

        # Precompute k-gram maps (dilakukan 1 kali per dokumen, menghemat ratusan ribu operasi)
        kgram_maps = {}
        for name in names:
            kgram_maps[name] = build_kgram_map(doc_database[name], k=k_val)

        # Cari pasangan yang belum ada di SQLite
        missing_pairs = []
        conn = sqlite3.connect(self.db_path)
        try:
            cur = conn.cursor()
            cur.execute("SELECT doc_a, doc_b FROM pairs WHERE k_val = ?", (k_val,))
            cached_pairs_set = set(cur.fetchall())

            all_pairs = list(combinations(names, 2))
            total_pairs = len(all_pairs)

            for da, db in all_pairs:
                pair_key = (da, db) if da < db else (db, da)
                if pair_key not in cached_pairs_set:
                    missing_pairs.append(pair_key)
        finally:
            conn.close()

        cached_count = total_pairs - len(missing_pairs)
        new_computed = 0

        if missing_pairs:
            batch = []
            conn = sqlite3.connect(self.db_path)
            total_missing = len(missing_pairs)
            step = max(1, total_missing // 50)
            try:
                for idx, (da, db) in enumerate(missing_pairs):
                    w_a, w_b = doc_database[da], doc_database[db]
                    m_a, m_b = kgram_maps[da], kgram_maps[db]
                    s_a, s_b, passages = calculate_turnitin_similarity(w_a, w_b, m_a, m_b, k=k_val)
                    max_s = max(s_a, s_b)
                    status = "PASS" if max_s <= threshold else "FAIL"
                    badge = turnitin_badge(max_s)

                    batch.append((
                        da, db, k_val, round(max_s, 2), round(s_a, 2), round(s_b, 2),
                        len(passages), json.dumps(passages[:max_passages]), badge, status
                    ))
                    new_computed += 1

                    if len(batch) >= 500:
                        conn.executemany("""
                            INSERT OR REPLACE INTO pairs
                            (doc_a, doc_b, k_val, max_score, score_a, score_b, matches, passages_json, badge, status)
                            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                        """, batch)
                        conn.commit()
                        batch = []

                    if progress_callback and ((idx + 1) % step == 0 or (idx + 1) == total_missing):
                        progress_callback(idx + 1, total_missing)

                if batch:
                    conn.executemany("""
                        INSERT OR REPLACE INTO pairs
                        (doc_a, doc_b, k_val, max_score, score_a, score_b, matches, passages_json, badge, status)
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    """, batch)
                    conn.commit()
            finally:
                conn.close()

        return total_pairs, cached_count, new_computed

    def get_results_dataframe(self, active_names, k_val=6):
        """Mengambil seluruh hasil analisis yang diurutkan dari skor tertinggi."""
        conn = sqlite3.connect(self.db_path)
        try:
            cur = conn.cursor()
            cur.execute("""
                SELECT doc_a, doc_b, max_score, status, badge, matches, passages_json, score_a, score_b
                FROM pairs
                WHERE k_val = ?
                ORDER BY max_score DESC
            """, (k_val,))
            rows = cur.fetchall()

            cur.execute("SELECT filename, word_count FROM documents")
            doc_words = dict(cur.fetchall())
        finally:
            conn.close()

        names_set = set(active_names)

        filtered_rows = []
        for r in rows:
            if r[0] in names_set and r[1] in names_set:
                passages = json.loads(r[6])
                filtered_rows.append({
                    "Dokumen 1": r[0],
                    "Dokumen 2": r[1],
                    "Turnitin Max Score (%)": r[2],
                    "Status Kelulusan": r[3],
                    "Kategori Turnitin": r[4],
                    "Doc 1 Cocok di Doc 2 (%)": r[7],
                    "Doc 2 Cocok di Doc 1 (%)": r[8],
                    "Total Kata Doc 1": doc_words.get(r[0], 0),
                    "Total Kata Doc 2": doc_words.get(r[1], 0),
                    "Jumlah Blok Teks Cocok": r[5],
                    "Matches": r[5],
                    "Passages": passages,
                    "Score A": r[7],
                    "Score B": r[8]
                })

        return pd.DataFrame(filtered_rows)

    def cleanup_deleted_documents(self, active_filenames):
        """Menghapus dokumen dan pasangannya dari cache jika file sudah tidak ada di disk."""
        conn = sqlite3.connect(self.db_path)
        try:
            cur = conn.cursor()
            cur.execute("SELECT filename FROM documents")
            cached_docs = [r[0] for r in cur.fetchall()]
            active_set = set(active_filenames)
            pruned_count = 0
            for d in cached_docs:
                if d not in active_set:
                    cur.execute("DELETE FROM documents WHERE filename = ?", (d,))
                    cur.execute("DELETE FROM pairs WHERE doc_a = ? OR doc_b = ?", (d, d))
                    pruned_count += 1
            conn.commit()
            return pruned_count
        finally:
            conn.close()


In [ ]:
# @title 3. Download & Sinkronisasi Google Drive (Bypass Limit 50 File)
import io
import time
import os
import re

def extract_folder_id(url):
    match = re.search(r"folders/([a-zA-Z0-9_-]+)", url)
    if match: return match.group(1)
    match = re.search(r"id=([a-zA-Z0-9_-]+)", url)
    if match: return match.group(1)
    return url.strip()

def upload_file_to_drive(file_path, folder_id, drive_service=None, log_callback=print):
    """Mengunggah atau memperbarui file ke folder Google Drive."""
    if not os.path.exists(file_path):
        return False
    file_name = os.path.basename(file_path)
    if not drive_service:
        try:
            from google.colab import auth
            auth.authenticate_user()
            from googleapiclient.discovery import build
            drive_service = build('drive', 'v3')
        except Exception:
            try:
                from googleapiclient.discovery import build
                drive_service = build('drive', 'v3')
            except Exception as e:
                log_callback(f"ℹ️ Google Drive upload dilewati: {e}")
                return False

    try:
        from googleapiclient.http import MediaFileUpload
        query = f"'{folder_id}' in parents and trashed = false and name = '{file_name}'"
        res = drive_service.files().list(q=query, fields="files(id, name)").execute()
        existing_files = res.get('files', [])
        media = MediaFileUpload(file_path, resumable=True)

        if existing_files:
            file_id = existing_files[0]['id']
            drive_service.files().update(
                fileId=file_id,
                media_body=media
            ).execute()
            log_callback(f"☁️ Berhasil memperbarui file di Google Drive: {file_name}")
        else:
            metadata = {
                'name': file_name,
                'parents': [folder_id]
            }
            drive_service.files().create(
                body=metadata,
                media_body=media,
                fields='id'
            ).execute()
            log_callback(f"☁️ Berhasil mengunggah file baru ke Google Drive: {file_name}")
        return True
    except Exception as e:
        log_callback(f"⚠️ Gagal mengunggah {file_name} ke Google Drive: {e}")
        return False

def sync_drive_folder(folder_url_or_id, destination="./dokumen_tugas_p3md", log_callback=print, progress_callback=None):
    """
    Sinkronisasi Google Drive dengan pagination (pageSize=1000) dan skip file lokal.
    Otomatis mendownload similarity_cache.db dan dokumen naskah tugas baru.
    """
    os.makedirs(destination, exist_ok=True)
    folder_id = extract_folder_id(folder_url_or_id)
    if not folder_id:
        log_callback("⚠️ URL Google Drive tidak valid!")
        return 0, 0

    drive_service = None
    try:
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        drive_service = build('drive', 'v3')
    except Exception as e:
        try:
            from googleapiclient.discovery import build
            drive_service = build('drive', 'v3')
        except Exception as e2:
            log_callback(f"⚠️ Error inisialisasi Google Drive: {e} / {e2}")
            return 0, 0

    if progress_callback:
        progress_callback(0, 1, "Membaca daftar file dari Google Drive...")
    log_callback("🔍 Mengambil daftar seluruh dokumen dari Google Drive...")
    query = f"'{folder_id}' in parents and trashed = false and mimeType != 'application/vnd.google-apps.folder'"
    items = []
    page_token = None

    # Pagination loop untuk bypass limit 50 / 100 file
    while True:
        try:
            results = drive_service.files().list(
                q=query,
                pageSize=1000,
                fields="nextPageToken, files(id, name, mimeType, size)",
                pageToken=page_token
            ).execute()
            items.extend(results.get('files', []))
            page_token = results.get('nextPageToken', None)
            if not page_token:
                break
        except Exception as e:
            log_callback(f"⚠️ Gagal mendapatkan daftar file: {e}")
            break

    log_callback(f"📂 Ditemukan {len(items)} file di folder Google Drive.")

    from googleapiclient.http import MediaIoBaseDownload

    # 1. Cek & sinkronisasi file cache database atau laporan Excel dari Google Drive
    for sync_fname in ["similarity_cache.db", "Turnitin_Similarity_Report_P3MD.xlsx"]:
        remote_file = next((it for it in items if it['name'] == sync_fname), None)
        if remote_file:
            local_fpath = os.path.join(destination, sync_fname)
            rem_size = int(remote_file.get('size', 0))
            if not os.path.exists(local_fpath):
                log_callback(f"📦 Mengunduh {sync_fname} dari Google Drive...")
                try:
                    req = drive_service.files().get_media(fileId=remote_file['id'])
                    with io.FileIO(local_fpath, 'wb') as fh:
                        dl_tool = MediaIoBaseDownload(fh, req)
                        d_flag = False
                        while d_flag is False:
                            _, d_flag = dl_tool.next_chunk()
                    log_callback(f"✅ {sync_fname} berhasil disinkronkan dari Google Drive!")
                except Exception as e:
                    log_callback(f"⚠️ Gagal mengunduh {sync_fname}: {e}")

    downloaded = 0
    skipped = 0
    supported_exts = {".docx", ".pdf", ".txt"}
    eligible_items = [
        item for item in items 
        if os.path.splitext(item['name'])[1].lower() in supported_exts and not item['name'].startswith("~")
    ]
    total_eligible = len(eligible_items)

    for idx, item in enumerate(eligible_items):
        file_id = item['id']
        file_name = item['name']
        local_path = os.path.join(destination, file_name)
        remote_size = int(item.get('size', 0))

        # Cek apakah file sudah ada secara lokal dengan ukuran sama
        if os.path.exists(local_path) and remote_size > 0:
            if os.path.getsize(local_path) == remote_size:
                skipped += 1
                if progress_callback:
                    progress_callback(idx + 1, total_eligible, f"File sudah ada (dilewati): {file_name}")
                continue

        if progress_callback:
            progress_callback(idx + 1, total_eligible, f"Mengunduh ({downloaded + 1}): {file_name}")

        # Unduh file baru/berubah dengan retry
        success = False
        for attempt in range(3):
            try:
                request = drive_service.files().get_media(fileId=file_id)
                with io.FileIO(local_path, 'wb') as fh:
                    downloader = MediaIoBaseDownload(fh, request)
                    done = False
                    while done is False:
                        status, done = downloader.next_chunk()
                downloaded += 1
                log_callback(f"  📥 Diunduh ({downloaded}): {file_name}")
                success = True
                break
            except Exception as e:
                time.sleep(1 + attempt * 2)

        if not success:
            log_callback(f"  ⚠️ Gagal mengunduh: {file_name}")

    # 3. Prune file lokal lama yang sudah dihapus/direname di Google Drive (mencegah duplikat tugas)
    remote_names = {it['name'] for it in eligible_items}
    pruned = 0
    if os.path.exists(destination):
        for local_f in os.listdir(destination):
            ext = os.path.splitext(local_f)[1].lower()
            if ext in supported_exts and not local_f.startswith("~"):
                if local_f not in remote_names:
                    del_path = os.path.join(destination, local_f)
                    try:
                        os.remove(del_path)
                        pruned += 1
                        log_callback(f"  🗑️ Menghapus file lokal lama yang sudah dihapus di Drive: {local_f}")
                    except Exception:
                        pass
        if pruned > 0:
            log_callback(f"🧹 Membersihkan {pruned} file lokal yang tidak ada lagi di Google Drive.")

    log_callback(f"✅ Sinkronisasi selesai: {downloaded} file baru diunduh, {skipped} file sudah ada dilewati.")
    return downloaded, skipped


In [ ]:
# @title 4. Generator Laporan Excel Resmi Turnitin (Executive 6-Sheet Edition)
"""
report_generator.py
Generator Laporan Excel Resmi & Komprehensif Turnitin P3MD.
Mengubah perbandingan pasangan (hingga puluhan ribu baris) menjadi sajian data
yang sangat mudah dipahami oleh peserta individu maupun tim penilai/komandan.
"""

import os
import sys
import re
import time
if sys.stdout and hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# --- PALET WARNA & STYLE STANDAR ---
NAVY_HEADER_FILL = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
TEAL_HEADER_FILL = PatternFill(start_color="2C6B6F", end_color="2C6B6F", fill_type="solid")
DARK_RED_HEADER_FILL = PatternFill(start_color="842029", end_color="842029", fill_type="solid")
HEADER_FONT = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")

CARD_HEADER_FILL = PatternFill(start_color="D9E1E8", end_color="D9E1E8", fill_type="solid")
CARD_HEADER_FONT = Font(name="Segoe UI", size=10, bold=True, color="1F4E79")
CARD_VALUE_FONT = Font(name="Segoe UI", size=18, bold=True, color="1F4E79")
CARD_SUB_FONT = Font(name="Segoe UI", size=9, italic=True, color="555555")

# Status Kelulusan
PASS_FILL = PatternFill(start_color="D4EDDA", end_color="D4EDDA", fill_type="solid")
PASS_FONT = Font(name="Segoe UI", size=10, bold=True, color="155724")
FAIL_FILL = PatternFill(start_color="F8D7DA", end_color="F8D7DA", fill_type="solid")
FAIL_FONT = Font(name="Segoe UI", size=10, bold=True, color="721C24")

# Zebra striping
ZEBRA_FILL = PatternFill(start_color="F9FAFB", end_color="F9FAFB", fill_type="solid")

# Borders
THIN_GRAY = Side(style="thin", color="D3D3D3")
BORDER_BOX = Border(left=THIN_GRAY, right=THIN_GRAY, top=THIN_GRAY, bottom=THIN_GRAY)
BORDER_TOP_BOTTOM = Border(top=THIN_GRAY, bottom=THIN_GRAY)
DOUBLE_BOTTOM = Side(style="double", color="1F4E79")
BORDER_CARD_BOTTOM = Border(left=THIN_GRAY, right=THIN_GRAY, top=THIN_GRAY, bottom=DOUBLE_BOTTOM)

# Shared Alignments & Number Formats (pre-instantiated for 30x faster generation)
ALIGN_CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
ALIGN_CENTER_NOWRAP = Alignment(horizontal="center", vertical="center")
ALIGN_RIGHT = Alignment(horizontal="right", vertical="center")
ALIGN_LEFT = Alignment(horizontal="left", vertical="center")
ALIGN_LEFT_WRAP = Alignment(horizontal="left", vertical="center", wrap_text=True)

NUM_FMT_PERCENT = '0.00"%"'
NUM_FMT_INT = '#,##0'

# Badges Turnitin
BADGE_STYLES = {
    "Blue": (PatternFill(start_color="CCE5FF", end_color="CCE5FF", fill_type="solid"), Font(name="Segoe UI", size=10, bold=True, color="004085")),
    "Green": (PatternFill(start_color="D4EDDA", end_color="D4EDDA", fill_type="solid"), Font(name="Segoe UI", size=10, bold=True, color="155724")),
    "Yellow": (PatternFill(start_color="FFF3CD", end_color="FFF3CD", fill_type="solid"), Font(name="Segoe UI", size=10, bold=True, color="856404")),
    "Orange": (PatternFill(start_color="FFE5D0", end_color="FFE5D0", fill_type="solid"), Font(name="Segoe UI", size=10, bold=True, color="A04000")),
    "Red": (PatternFill(start_color="F8D7DA", end_color="F8D7DA", fill_type="solid"), Font(name="Segoe UI", size=10, bold=True, color="721C24")),
}

def get_turnitin_tier(score):
    if score == 0:
        return "Blue", "🔵 Blue (0%)"
    elif score < 25:
        return "Green", "🟢 Green (1-24%)"
    elif score < 50:
        return "Yellow", "🟡 Yellow (25-49%)"
    elif score < 75:
        return "Orange", "🟠 Orange (50-74%)"
    else:
        return "Red", "🔴 Red (75-100%)"

def style_header_row(ws, row_idx, num_cols, fill=NAVY_HEADER_FILL, font=HEADER_FONT):
    for c in range(1, num_cols + 1):
        cell = ws.cell(row=row_idx, column=c)
        cell.fill = fill
        cell.font = font
        cell.alignment = ALIGN_CENTER
        cell.border = BORDER_BOX
    ws.row_dimensions[row_idx].height = 28

def auto_fit_columns(ws, min_w=12, max_w=65, sample_rows=100):
    """
    Menyesuaikan lebar kolom secara cerdas dan cepat.
    Menggunakan sampling hingga 100 baris pertama untuk menghindari lag pada puluhan ribu baris.
    """
    max_r = min(ws.max_row, sample_rows)
    for col_idx in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_idx)
        max_len = 0
        for r in range(1, max_r + 1):
            val = ws.cell(row=r, column=col_idx).value
            if val is not None:
                s_val = str(val)
                if "\n" in s_val:
                    max_len = max(max_len, max(len(l) for l in s_val.split("\n")))
                else:
                    max_len = max(max_len, len(s_val))
        ws.column_dimensions[col_letter].width = min(max(max_len + 3, min_w), max_w)


def compute_leaderboard(df_pairs, threshold=15.0):
    """
    Menghitung rekapitulasi per dokumen (1 baris per dokumen).
    Memproses setiap dokumen terhadap seluruh pasangannya dalam cohort.
    Menggunakan to_dict('records') untuk efisiensi O(N).
    """
    records = df_pairs.to_dict('records') if hasattr(df_pairs, 'to_dict') else df_pairs
    all_docs = set()
    doc_matches = {}
    doc_word_counts = {}

    for r in records:
        d1 = r.get("Dokumen 1")
        d2 = r.get("Dokumen 2")
        if not d1 or not d2:
            continue
        all_docs.add(d1)
        all_docs.add(d2)

        max_s = float(r.get("Turnitin Max Score (%)") or r.get("Max Score") or 0.0)
        status = r.get("Status Kelulusan") or r.get("Status") or "PASS"
        blocks = r.get("Jumlah Blok Teks Cocok") or r.get("Matches") or 0

        # Doc 1 stats
        w1 = r.get("Total Kata Doc 1") or r.get("Words A") or 0
        if w1 and d1 not in doc_word_counts:
            doc_word_counts[d1] = w1

        # Doc 2 stats
        w2 = r.get("Total Kata Doc 2") or r.get("Words B") or 0
        if w2 and d2 not in doc_word_counts:
            doc_word_counts[d2] = w2

        if d1 not in doc_matches:
            doc_matches[d1] = []
        if d2 not in doc_matches:
            doc_matches[d2] = []

        # Record match from d1 perspective
        doc_matches[d1].append({
            "partner": d2,
            "score": max_s,
            "status": status,
            "blocks": blocks
        })
        # Record match from d2 perspective
        doc_matches[d2].append({
            "partner": d1,
            "score": max_s,
            "status": status,
            "blocks": blocks
        })

    leaderboard = []
    for doc, matches in doc_matches.items():
        if not matches:
            continue
        matches_sorted = sorted(matches, key=lambda x: x["score"], reverse=True)
        top_match = matches_sorted[0]
        second_match = matches_sorted[1] if len(matches_sorted) > 1 else None

        worst_score = top_match["score"]
        overall_status = "FAIL" if worst_score > threshold else "PASS"
        fail_partners_count = sum(1 for m in matches if m["score"] > threshold)
        avg_score = sum(m["score"] for m in matches) / len(matches)
        tier_key, tier_label = get_turnitin_tier(worst_score)

        leaderboard.append({
            "Dokumen": doc,
            "Status Kelulusan": overall_status,
            "Skor Tertinggi (%)": worst_score,
            "Kategori Turnitin": tier_label,
            "Tier Key": tier_key,
            "Top Matched Document": top_match["partner"],
            "Top Match Score (%)": top_match["score"],
            "Top Match Blocks": top_match["blocks"],
            "2nd Matched Document": second_match["partner"] if second_match else "-",
            "2nd Match Score (%)": second_match["score"] if second_match else 0.0,
            "Rata-rata Similaritas Cohort (%)": round(avg_score, 2),
            "Jumlah Pasangan > Batas": fail_partners_count,
            "Total Kata": doc_word_counts.get(doc, "-")
        })

    # Urutkan: FAIL lebih dulu, lalu skor tertinggi descending
    leaderboard.sort(key=lambda x: (0 if x["Status Kelulusan"] == "FAIL" else 1, -x["Skor Tertinggi (%)"]))
    return leaderboard


def build_two_way_comparisons(df_pairs):
    """
    Membangun tabel komparasi dua arah (Target vs Pembanding).
    Memudahkan peserta memfilter 1 dokumen target untuk melihat seluruh lawannya.
    Menggunakan to_dict('records') untuk kecepatan maksimal.
    """
    records = df_pairs.to_dict('records') if hasattr(df_pairs, 'to_dict') else df_pairs
    two_way = []
    for r in records:
        d1 = r.get("Dokumen 1")
        d2 = r.get("Dokumen 2")
        if not d1 or not d2:
            continue
        max_s = float(r.get("Turnitin Max Score (%)") or r.get("Max Score") or 0.0)
        status = r.get("Status Kelulusan") or r.get("Status") or "PASS"
        badge = r.get("Kategori Turnitin") or r.get("Badge") or "-"
        blocks = r.get("Jumlah Blok Teks Cocok") or r.get("Matches") or 0

        s1 = float(r.get("Doc 1 Cocok di Doc 2 (%)") or r.get("Score A") or max_s)
        s2 = float(r.get("Doc 2 Cocok di Doc 1 (%)") or r.get("Score B") or max_s)

        # Baris dari perspektif Dokumen 1
        two_way.append({
            "Dokumen Target": d1,
            "Dokumen Pembanding": d2,
            "Turnitin Max Score (%)": max_s,
            "Status Kelulusan": status,
            "Kategori Turnitin": badge,
            "Kemiripan Target di Pembanding (%)": s1,
            "Kemiripan Pembanding di Target (%)": s2,
            "Jumlah Blok Cocok": blocks
        })
        # Baris dari perspektif Dokumen 2
        two_way.append({
            "Dokumen Target": d2,
            "Dokumen Pembanding": d1,
            "Turnitin Max Score (%)": max_s,
            "Status Kelulusan": status,
            "Kategori Turnitin": badge,
            "Kemiripan Target di Pembanding (%)": s2,
            "Kemiripan Pembanding di Target (%)": s1,
            "Jumlah Blok Cocok": blocks
        })

    two_way.sort(key=lambda x: (x["Dokumen Target"], -x["Turnitin Max Score (%)"]))
    return two_way


def generate_excel_report(df_results, total_docs, min_words=6, commander_threshold=17.0, pass_threshold=15.0, 
                          matched_passages=None, output_file="Turnitin_Similarity_Report_P3MD.xlsx"):
    """
    Fungsi utama pembuatan file Excel 6-Sheet Executive Report.
    """
    wb = openpyxl.Workbook()

    # 1. Hitung Leaderboard Per Peserta
    leaderboard = compute_leaderboard(df_results, threshold=pass_threshold)
    actual_total_docs = len(leaderboard) if leaderboard else total_docs
    total_pairs = len(df_results)
    
    passed_docs = sum(1 for d in leaderboard if d["Status Kelulusan"] == "PASS")
    failed_docs = sum(1 for d in leaderboard if d["Status Kelulusan"] == "FAIL")
    pass_rate = (passed_docs / actual_total_docs * 100) if actual_total_docs > 0 else 0.0

    # Pasangan yang melebihi batas (FAIL)
    score_col = "Turnitin Max Score (%)" if "Turnitin Max Score (%)" in df_results.columns else "Max Score"
    df_flagged = df_results[df_results[score_col].astype(float) > pass_threshold]
    flagged_pairs_count = len(df_flagged)

    # -------------------------------------------------------------------------
    # SHEET 1: 📊 Dashboard Eksekutif
    # -------------------------------------------------------------------------
    ws_dash = wb.active
    ws_dash.title = "📊 Dashboard Eksekutif"
    ws_dash.views.sheetView[0].showGridLines = True

    # Title Banner
    ws_dash.merge_cells("A1:G1")
    title_cell = ws_dash["A1"]
    title_cell.value = "🔍 LAPORAN EKSEKUTIF SIMILARITAS TURNITIN - DOKUMEN P3MD"
    title_cell.fill = NAVY_HEADER_FILL
    title_cell.font = Font(name="Segoe UI", size=14, bold=True, color="FFFFFF")
    title_cell.alignment = Alignment(horizontal="center", vertical="center")
    ws_dash.row_dimensions[1].height = 36

    # Subtitle / Info Bar
    ws_dash.merge_cells("A2:G2")
    sub_cell = ws_dash["A2"]
    sub_cell.value = f"Waktu Pembuatan: {time.strftime('%d %B %Y, %H:%M:%S')}  |  Ambang Batas Kelulusan: {pass_threshold}% (Safety Cushion)  |  Batas Resmi Komandan: {commander_threshold}%  |  Min Kata (k-gram): {min_words}"
    sub_cell.fill = CARD_HEADER_FILL
    sub_cell.font = Font(name="Segoe UI", size=9, bold=True, color="1F4E79")
    sub_cell.alignment = Alignment(horizontal="center", vertical="center")
    ws_dash.row_dimensions[2].height = 22

    # KPI Summary Cards (Row 4 to Row 6)
    kpis = [
        ("A", "B", "TOTAL DOKUMEN", f"{actual_total_docs} File", f"Dievaluasi dalam cohort", "1F4E79"),
        ("C", "C", "DOKUMEN LULUS", f"{passed_docs} ({pass_rate:.1f}%)", f"Skor maks <= {pass_threshold}%", "155724"),
        ("D", "D", "MELEBIHI BATAS", f"{failed_docs} File", f"Perlu perbaikan / remedial", "721C24"),
        ("E", "F", "TOTAL PASANGAN DIUJI", f"{total_pairs:,} Pasang", f"Kombinasi n*(n-1)/2", "1F4E79"),
        ("G", "G", "PASANGAN GAGAL", f"{flagged_pairs_count} Pasang", f"Skor > {pass_threshold}%", "721C24")
    ]

    for start_col, end_col, card_title, card_val, card_sub, color_code in kpis:
        c_top = ws_dash[f"{start_col}4"]
        c_val = ws_dash[f"{start_col}5"]
        c_sub = ws_dash[f"{start_col}6"]

        if start_col != end_col:
            ws_dash.merge_cells(f"{start_col}4:{end_col}4")
            ws_dash.merge_cells(f"{start_col}5:{end_col}5")
            ws_dash.merge_cells(f"{start_col}6:{end_col}6")

        c_top.value = card_title
        c_top.fill = CARD_HEADER_FILL
        c_top.font = CARD_HEADER_FONT
        c_top.alignment = Alignment(horizontal="center", vertical="center")
        c_top.border = BORDER_BOX

        c_val.value = card_val
        c_val.font = Font(name="Segoe UI", size=16, bold=True, color=color_code)
        c_val.alignment = Alignment(horizontal="center", vertical="center")
        c_val.border = BORDER_BOX

        c_sub.value = card_sub
        c_sub.font = CARD_SUB_FONT
        c_sub.alignment = Alignment(horizontal="center", vertical="center")
        c_sub.border = BORDER_CARD_BOTTOM

    ws_dash.row_dimensions[4].height = 20
    ws_dash.row_dimensions[5].height = 32
    ws_dash.row_dimensions[6].height = 18

    # Tier Breakdown Table (Row 8 to Row 15)
    ws_dash.cell(row=8, column=1, value="📊 DISTRIBUSI TIER WARNA STANDAR TURNITIN COHORT").font = Font(name="Segoe UI", size=11, bold=True, color="1F4E79")
    
    tier_headers = ["Kategori / Tier Turnitin", "Rentang Skor", "Jumlah Dokumen", "Persentase Dokumen", "Status Standar", "Tindakan Disarankan"]
    for idx, h in enumerate(tier_headers, 1):
        cell = ws_dash.cell(row=9, column=idx, value=h)
        cell.fill = NAVY_HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = BORDER_BOX
    ws_dash.row_dimensions[9].height = 24

    # Count distribution per tier based on leaderboard max scores
    tier_counts = {"Blue": 0, "Green": 0, "Yellow": 0, "Orange": 0, "Red": 0}
    for item in leaderboard:
        t_key = item["Tier Key"]
        tier_counts[t_key] = tier_counts.get(t_key, 0) + 1

    tier_rows_meta = [
        ("🔵 Blue (0%)", "0%", tier_counts["Blue"], "Aman Sempurna", "Lulus murni tanpa kesamaan verbatim.", "Blue"),
        ("🟢 Green (1-24%)", "1% - 24%", tier_counts["Green"], "Sangat Baik / Wajar", f"Lulus jika <= {pass_threshold}%. Kemiripan wajar format/istilah umum.", "Green"),
        ("🟡 Yellow (25-49%)", "25% - 49%", tier_counts["Yellow"], "Waspada / Mencurigakan", "Tidak Lulus. Terdapat bagian teks panjang yang sama.", "Yellow"),
        ("🟠 Orange (50-74%)", "50% - 74%", tier_counts["Orange"], "Risiko Tinggi", "Tidak Lulus. Mayoritas isi dokumen identik dengan peserta lain.", "Orange"),
        ("🔴 Red (75-100%)", "75% - 100%", tier_counts["Red"], "Plagiasi Berat", "Tidak Lulus. Dokumen duplikasi langsung / copy-paste penuh.", "Red")
    ]

    for r_idx, (t_name, t_range, t_count, t_status, t_action, t_key) in enumerate(tier_rows_meta, 10):
        pct = (t_count / actual_total_docs * 100) if actual_total_docs > 0 else 0.0
        ws_dash.cell(row=r_idx, column=1, value=t_name).font = Font(name="Segoe UI", size=10, bold=True)
        ws_dash.cell(row=r_idx, column=2, value=t_range).alignment = Alignment(horizontal="center")
        ws_dash.cell(row=r_idx, column=3, value=t_count).alignment = Alignment(horizontal="center")
        ws_dash.cell(row=r_idx, column=4, value=f"{pct:.1f}%").alignment = Alignment(horizontal="center")
        
        status_cell = ws_dash.cell(row=r_idx, column=5, value=t_status)
        status_cell.alignment = Alignment(horizontal="center")
        fill_badge, font_badge = BADGE_STYLES[t_key]
        status_cell.fill = fill_badge
        status_cell.font = font_badge

        ws_dash.cell(row=r_idx, column=6, value=t_action).font = Font(name="Segoe UI", size=9)

        for c in range(1, 7):
            ws_dash.cell(row=r_idx, column=c).border = BORDER_BOX
        ws_dash.row_dimensions[r_idx].height = 22

    # Panduan Penggunaan Workbook (Row 17 onwards)
    guide_row = 16
    ws_dash.cell(row=guide_row, column=1, value="📖 PANDUAN MEMBACA & MENGGUNAKAN LAPORAN INI").font = Font(name="Segoe UI", size=11, bold=True, color="1F4E79")
    
    guides = [
        ("1. Bagi Peserta Tugas / Penulis Dokumen:", 
         "Buka sheet '👤 Rekap Per Peserta'. Cukup cari nama Anda (Ctrl+F). Anda akan langsung melihat: Status Lulus/Gagal, Nilai Similaritas Tertinggi Anda, Dokumen Lawan yang paling mirip, dan Rata-rata Similaritas Cohort. Anda TIDAK PERLU memeriksa ribuan baris pasangan."),
        ("2. Memahami Nilai Similaritas:",
         f"Sistem Turnitin membandingkan naskah Anda terhadap seluruh peserta. Jika dokumen Anda memiliki kemiripan 35% dengan Si B, maka skor resmi Anda adalah 35% (status FAIL karena melebihi batas {pass_threshold}%). Rata-rata cohort (~2%) mencerminkan kemiripan template soal."),
        ("3. Bagi Komandan / Tim Penilai:",
         "Buka sheet '⚠️ Investigasi Plagiasi' untuk melihat langsung daftar pasangan yang melanggar batas kelulusan tanpa terganggu ribuan pasangan yang bersih. Buka sheet '📝 Detail Kalimat Identik' untuk memeriksa bukti kutipan kalimat verbatim yang sama."),
        ("4. Analisis Detail Pasangan Tertentu:",
         "Buka sheet '🔍 Cek Dokumen Individu' jika Anda ingin melihat peringkat perbandingan lengkap dari 1 dokumen spesifik terhadap seluruh 397 dokumen lainnya secara terurut.")
    ]

    for g_idx, (g_title, g_desc) in enumerate(guides, guide_row + 1):
        ws_dash.cell(row=g_idx, column=1, value=g_title).font = Font(name="Segoe UI", size=10, bold=True, color="1F4E79")
        ws_dash.merge_cells(f"B{g_idx}:G{g_idx}")
        desc_cell = ws_dash[f"B{g_idx}"]
        desc_cell.value = g_desc
        desc_cell.font = Font(name="Segoe UI", size=9)
        desc_cell.alignment = Alignment(wrap_text=True)
        ws_dash.row_dimensions[g_idx].height = 36

    auto_fit_columns(ws_dash, min_w=14, max_w=70)


    # -------------------------------------------------------------------------
    # SHEET 2: 👤 Rekap Per Peserta (Leaderboard) - 1 Baris Per Dokumen
    # -------------------------------------------------------------------------
    ws_lead = wb.create_sheet("👤 Rekap Per Peserta")
    ws_lead.views.sheetView[0].showGridLines = True
    
    lead_headers = [
        "Rank", "Nama Dokumen (Peserta)", "Status Kelulusan", "Skor Tertinggi (%)", 
        "Kategori Turnitin", "Pasangan Paling Mirip (Top Match)", "Skor Match #1 (%)", 
        "Pasangan Match #2", "Skor Match #2 (%)", "Rata-rata Similaritas Cohort (%)", 
        "Jumlah Pasangan > Batas", "Total Kata"
    ]
    ws_lead.append(lead_headers)
    style_header_row(ws_lead, 1, len(lead_headers), fill=NAVY_HEADER_FILL)

    for rank, item in enumerate(leaderboard, 1):
        r_num = ws_lead.max_row + 1
        ws_lead.append([
            rank,
            item["Dokumen"],
            item["Status Kelulusan"],
            item["Skor Tertinggi (%)"],
            item["Kategori Turnitin"],
            item["Top Matched Document"],
            item["Top Match Score (%)"],
            item["2nd Matched Document"],
            item["2nd Match Score (%)"],
            item["Rata-rata Similaritas Cohort (%)"],
            item["Jumlah Pasangan > Batas"],
            item["Total Kata"]
        ])

        # Style row
        row_cells = [ws_lead.cell(row=r_num, column=c) for c in range(1, len(lead_headers) + 1)]
        
        # Rank & numbers alignment
        row_cells[0].alignment = Alignment(horizontal="center")
        row_cells[2].alignment = Alignment(horizontal="center")
        row_cells[3].alignment = Alignment(horizontal="right")
        row_cells[4].alignment = Alignment(horizontal="center")
        row_cells[6].alignment = Alignment(horizontal="right")
        row_cells[8].alignment = Alignment(horizontal="right")
        row_cells[9].alignment = Alignment(horizontal="right")
        row_cells[10].alignment = Alignment(horizontal="center")
        row_cells[11].alignment = Alignment(horizontal="right")

        # Number formatting
        row_cells[3].number_format = '0.00"%"'
        row_cells[6].number_format = '0.00"%"'
        row_cells[8].number_format = '0.00"%"'
        row_cells[9].number_format = '0.00"%"'
        if isinstance(item["Total Kata"], (int, float)):
            row_cells[11].number_format = '#,##0'

        # Status styling
        if item["Status Kelulusan"] == "PASS":
            row_cells[2].fill = PASS_FILL
            row_cells[2].font = PASS_FONT
        else:
            row_cells[2].fill = FAIL_FILL
            row_cells[2].font = FAIL_FONT

        # Badge styling
        t_key = item["Tier Key"]
        f_badge, fnt_badge = BADGE_STYLES.get(t_key, (None, None))
        if f_badge:
            row_cells[4].fill = f_badge
            row_cells[4].font = fnt_badge

        # Zebra striping for neutral columns
        if rank % 2 == 0 and item["Status Kelulusan"] == "PASS":
            for c_idx in [1, 5, 7]:
                row_cells[c_idx].fill = ZEBRA_FILL

        for cell in row_cells:
            cell.border = BORDER_BOX
        ws_lead.row_dimensions[r_num].height = 20

    ws_lead.freeze_panes = "A2"
    ws_lead.auto_filter.ref = ws_lead.dimensions
    auto_fit_columns(ws_lead, min_w=10, max_w=55)


    # -------------------------------------------------------------------------
    # SHEET 3: ⚠️ Investigasi Plagiasi (Hanya Pasangan Melebihi Batas)
    # -------------------------------------------------------------------------
    ws_flag = wb.create_sheet("⚠️ Investigasi Plagiasi")
    ws_flag.views.sheetView[0].showGridLines = True

    flag_headers = [
        "No.", "Dokumen 1", "Dokumen 2", "Turnitin Max Score (%)", "Status", 
        "Kategori Turnitin", "Doc 1 Cocok di Doc 2 (%)", "Doc 2 Cocok di Doc 1 (%)", 
        "Total Kata Doc 1", "Total Kata Doc 2", "Jumlah Blok Cocok", "Indikasi & Dugaan Hubungan"
    ]
    ws_flag.append(flag_headers)
    style_header_row(ws_flag, 1, len(flag_headers), fill=DARK_RED_HEADER_FILL)

    flag_idx = 0
    df_records_all = df_results.to_dict('records') if hasattr(df_results, 'to_dict') else df_results
    for r in df_records_all:
        max_s = float(r.get("Turnitin Max Score (%)") or r.get("Max Score") or 0.0)
        if max_s <= pass_threshold:
            continue

        flag_idx += 1
        r_num = ws_flag.max_row + 1

        d1 = r.get("Dokumen 1")
        d2 = r.get("Dokumen 2")
        status = r.get("Status Kelulusan") or r.get("Status") or "FAIL"
        badge = r.get("Kategori Turnitin") or r.get("Badge") or "-"
        s1 = float(r.get("Doc 1 Cocok di Doc 2 (%)") or r.get("Score A") or max_s)
        s2 = float(r.get("Doc 2 Cocok di Doc 1 (%)") or r.get("Score B") or max_s)
        w1 = r.get("Total Kata Doc 1") or r.get("Words A") or 0
        w2 = r.get("Total Kata Doc 2") or r.get("Words B") or 0
        blocks = r.get("Jumlah Blok Teks Cocok") or r.get("Matches") or 0

        # Analisis rasio & arah dugaan
        if abs(s1 - s2) > 10 and w1 and w2:
            if s1 > s2:
                indikasi = f"Dokumen 1 ({w1:,} kata) memuat porsi besar teks dari Dokumen 2 ({w2:,} kata). Indikasi Dokumen 1 menyalin sebagian Dokumen 2."
            else:
                indikasi = f"Dokumen 2 ({w2:,} kata) memuat porsi besar teks dari Dokumen 1 ({w1:,} kata). Indikasi Dokumen 2 menyalin sebagian Dokumen 1."
        else:
            indikasi = "Kemiripan simetris dua arah dalam skala signifikan. Indikasi pengerjaan bersama atau sumber bahan yang sama."

        ws_flag.append([
            flag_idx, d1, d2, max_s, status, badge, s1, s2, w1, w2, blocks, indikasi
        ])

        # Fast styling
        row_cells = [ws_flag.cell(row=r_num, column=c) for c in range(1, len(flag_headers) + 1)]
        row_cells[0].alignment = ALIGN_CENTER_NOWRAP
        row_cells[3].alignment = ALIGN_RIGHT
        row_cells[3].number_format = NUM_FMT_PERCENT
        row_cells[4].alignment = ALIGN_CENTER_NOWRAP
        row_cells[4].fill = FAIL_FILL
        row_cells[4].font = FAIL_FONT
        row_cells[5].alignment = ALIGN_CENTER_NOWRAP
        row_cells[6].alignment = ALIGN_RIGHT
        row_cells[6].number_format = NUM_FMT_PERCENT
        row_cells[7].alignment = ALIGN_RIGHT
        row_cells[7].number_format = NUM_FMT_PERCENT
        row_cells[8].alignment = ALIGN_RIGHT
        row_cells[9].alignment = ALIGN_RIGHT
        row_cells[10].alignment = ALIGN_CENTER_NOWRAP
        row_cells[11].alignment = ALIGN_LEFT_WRAP

        for cell in row_cells:
            cell.border = BORDER_BOX
        ws_flag.row_dimensions[r_num].height = 30

    if flag_idx == 0:
        ws_flag.append(["-", "Selamat! Tidak ada pasangan dokumen yang melebihi batas kelulusan.", "", "", "ALL PASS", "", "", "", "", "", "", ""])
        ws_flag.cell(row=2, column=2).font = Font(name="Segoe UI", size=11, bold=True, color="155724")

    ws_flag.freeze_panes = "A2"
    ws_flag.auto_filter.ref = ws_flag.dimensions
    auto_fit_columns(ws_flag, min_w=10, max_w=65)


    # -------------------------------------------------------------------------
    # SHEET 4: 🔍 Cek Dokumen Individu (Two-Way Inspector View)
    # -------------------------------------------------------------------------
    ws_inspect = wb.create_sheet("🔍 Cek Dokumen Individu")
    ws_inspect.views.sheetView[0].showGridLines = True
    ws_inspect.sheet_format.defaultRowHeight = 19

    inspect_headers = [
        "Dokumen Target (Filter Nama Anda Di Sini)", "Dokumen Pembanding", 
        "Turnitin Max Score (%)", "Status Kelulusan", "Kategori Turnitin", 
        "Kemiripan Target di Pembanding (%)", "Kemiripan Pembanding di Target (%)", "Jumlah Blok Cocok"
    ]
    ws_inspect.append(inspect_headers)
    style_header_row(ws_inspect, 1, len(inspect_headers), fill=TEAL_HEADER_FILL)

    two_way_data = build_two_way_comparisons(df_results)
    for r_idx, r_data in enumerate(two_way_data, 2):
        status = r_data["Status Kelulusan"]
        ws_inspect.append([
            r_data["Dokumen Target"],
            r_data["Dokumen Pembanding"],
            r_data["Turnitin Max Score (%)"],
            status,
            r_data["Kategori Turnitin"],
            r_data["Kemiripan Target di Pembanding (%)"],
            r_data["Kemiripan Pembanding di Target (%)"],
            r_data["Jumlah Blok Cocok"]
        ])

        c_score = ws_inspect.cell(row=r_idx, column=3)
        c_score.alignment = ALIGN_RIGHT
        c_score.number_format = NUM_FMT_PERCENT

        c_status = ws_inspect.cell(row=r_idx, column=4)
        c_status.alignment = ALIGN_CENTER_NOWRAP

        c_badge = ws_inspect.cell(row=r_idx, column=5)
        c_badge.alignment = ALIGN_CENTER_NOWRAP

        c_s1 = ws_inspect.cell(row=r_idx, column=6)
        c_s1.alignment = ALIGN_RIGHT
        c_s1.number_format = NUM_FMT_PERCENT

        c_s2 = ws_inspect.cell(row=r_idx, column=7)
        c_s2.alignment = ALIGN_RIGHT
        c_s2.number_format = NUM_FMT_PERCENT

        c_blocks = ws_inspect.cell(row=r_idx, column=8)
        c_blocks.alignment = ALIGN_CENTER_NOWRAP

        if status == "FAIL":
            c_status.fill = FAIL_FILL
            c_status.font = FAIL_FONT

    ws_inspect.freeze_panes = "A2"
    ws_inspect.auto_filter.ref = ws_inspect.dimensions
    auto_fit_columns(ws_inspect, min_w=12, max_w=55)


    # -------------------------------------------------------------------------
    # SHEET 5: 📝 Detail Kalimat Identik
    # -------------------------------------------------------------------------
    ws_pass = wb.create_sheet("📝 Detail Kalimat Identik")
    ws_pass.views.sheetView[0].showGridLines = True

    pass_headers = [
        "Dokumen 1", "Dokumen 2", "Status Pasangan", "Skor Max (%)", 
        "Perkiraan Jumlah Kata Cocok", "Potongan Kalimat Identik (Verbatim)"
    ]
    ws_pass.append(pass_headers)
    style_header_row(ws_pass, 1, len(pass_headers), fill=NAVY_HEADER_FILL)

    # Populate matched passages with priority given to FAIL pairs
    if matched_passages:
        passages_sorted = sorted(
            matched_passages, 
            key=lambda x: (0 if x.get("status") == "FAIL" else 1, -float(str(x.get("score", 0)).replace("%", "")))
        )
        for p_item in passages_sorted:
            r_num = ws_pass.max_row + 1
            text_val = p_item.get("text", "")
            word_cnt = len(text_val.split())
            score_num = float(str(p_item.get("score", 0)).replace("%", ""))

            ws_pass.append([
                p_item.get("doc1", "-"),
                p_item.get("doc2", "-"),
                p_item.get("status", "PASS"),
                score_num,
                word_cnt,
                text_val
            ])

            row_cells = [ws_pass.cell(row=r_num, column=c) for c in range(1, len(pass_headers) + 1)]
            row_cells[2].alignment = ALIGN_CENTER_NOWRAP
            row_cells[3].alignment = ALIGN_RIGHT
            row_cells[3].number_format = NUM_FMT_PERCENT
            row_cells[4].alignment = ALIGN_CENTER_NOWRAP
            row_cells[5].alignment = ALIGN_LEFT_WRAP

            if p_item.get("status") == "FAIL":
                row_cells[2].fill = FAIL_FILL
                row_cells[2].font = FAIL_FONT
            else:
                row_cells[2].fill = PASS_FILL
                row_cells[2].font = PASS_FONT

            for cell in row_cells:
                cell.border = BORDER_BOX
            ws_pass.row_dimensions[r_num].height = 24

    ws_pass.freeze_panes = "A2"
    ws_pass.auto_filter.ref = ws_pass.dimensions
    auto_fit_columns(ws_pass, min_w=12, max_w=75)


    # -------------------------------------------------------------------------
    # SHEET 6: 📋 Semua Pasangan (Arsip)
    # -------------------------------------------------------------------------
    ws_all = wb.create_sheet("📋 Semua Pasangan (Arsip)")
    ws_all.views.sheetView[0].showGridLines = True
    ws_all.sheet_format.defaultRowHeight = 19

    all_headers = [
        "No.", "Dokumen 1", "Dokumen 2", "Turnitin Max Score (%)", "Status Kelulusan", 
        "Kategori Turnitin", "Doc 1 Cocok di Doc 2 (%)", "Doc 2 Cocok di Doc 1 (%)", 
        "Total Kata Doc 1", "Total Kata Doc 2", "Jumlah Blok Teks Cocok"
    ]
    ws_all.append(all_headers)
    style_header_row(ws_all, 1, len(all_headers), fill=NAVY_HEADER_FILL)

    for idx, r in enumerate(df_records_all, 1):
        r_num = idx + 1
        max_s = float(r.get("Turnitin Max Score (%)") or r.get("Max Score") or 0.0)
        status = r.get("Status Kelulusan") or r.get("Status") or "PASS"
        badge = r.get("Kategori Turnitin") or r.get("Badge") or "-"
        s1 = float(r.get("Doc 1 Cocok di Doc 2 (%)") or r.get("Score A") or max_s)
        s2 = float(r.get("Doc 2 Cocok di Doc 1 (%)") or r.get("Score B") or max_s)
        w1 = r.get("Total Kata Doc 1") or r.get("Words A") or "-"
        w2 = r.get("Total Kata Doc 2") or r.get("Words B") or "-"
        blocks = r.get("Jumlah Blok Teks Cocok") or r.get("Matches") or 0

        ws_all.append([
            idx, r.get("Dokumen 1"), r.get("Dokumen 2"), max_s, status, badge, s1, s2, w1, w2, blocks
        ])

        c_no = ws_all.cell(row=r_num, column=1)
        c_no.alignment = ALIGN_CENTER_NOWRAP

        c_score = ws_all.cell(row=r_num, column=4)
        c_score.alignment = ALIGN_RIGHT
        c_score.number_format = NUM_FMT_PERCENT

        c_status = ws_all.cell(row=r_num, column=5)
        c_status.alignment = ALIGN_CENTER_NOWRAP

        c_badge = ws_all.cell(row=r_num, column=6)
        c_badge.alignment = ALIGN_CENTER_NOWRAP

        c_s1 = ws_all.cell(row=r_num, column=7)
        c_s1.alignment = ALIGN_RIGHT
        c_s1.number_format = NUM_FMT_PERCENT

        c_s2 = ws_all.cell(row=r_num, column=8)
        c_s2.alignment = ALIGN_RIGHT
        c_s2.number_format = NUM_FMT_PERCENT

        c_w1 = ws_all.cell(row=r_num, column=9)
        c_w1.alignment = ALIGN_RIGHT
        if isinstance(w1, (int, float)):
            c_w1.number_format = NUM_FMT_INT

        c_w2 = ws_all.cell(row=r_num, column=10)
        c_w2.alignment = ALIGN_RIGHT
        if isinstance(w2, (int, float)):
            c_w2.number_format = NUM_FMT_INT

        c_blocks = ws_all.cell(row=r_num, column=11)
        c_blocks.alignment = ALIGN_CENTER_NOWRAP

        if status == "FAIL":
            c_status.fill = FAIL_FILL
            c_status.font = FAIL_FONT

    ws_all.freeze_panes = "A2"
    ws_all.auto_filter.ref = ws_all.dimensions
    auto_fit_columns(ws_all, min_w=10, max_w=50)


    # -------------------------------------------------------------------------
    # SHEET 7: ⚙️ Parameter Analisis
    # -------------------------------------------------------------------------
    ws_meta = wb.create_sheet("⚙️ Parameter Analisis")
    ws_meta.views.sheetView[0].showGridLines = True

    meta_headers = ["Parameter Sistem", "Nilai Konfigurasi", "Keterangan Standar Turnitin"]
    ws_meta.append(meta_headers)
    style_header_row(ws_meta, 1, len(meta_headers), fill=NAVY_HEADER_FILL)

    metadata_rows = [
        ("Ambang Batas Resmi Komandan (Official Threshold)", f"{commander_threshold}%", "Batas maksimal toleransi similaritas yang diizinkan komandan."),
        ("Ambang Batas Pengujian (Safety Cushion Threshold)", f"{pass_threshold}%", "Batas aman sistem (cushioning) untuk mendeteksi potensi pelanggaran."),
        ("Minimum Consecutive Words (k-gram Shingling)", f"{min_words} kata berurutan", "Standar resmi Turnitin untuk mendeteksi kesamaan frasa verbatim."),
        ("Eksklusi Kutipan (Filter Quotes)", "True (Aktif)", "Mengabaikan teks di dalam tanda petik ganda (\"...\" / “...”)."),
        ("Eksklusi Daftar Pustaka (Filter Bibliography)", "True (Aktif)", "Mengabaikan bab Daftar Pustaka, References, dan Rujukan."),
        ("Total Dokumen Dianalisis", f"{actual_total_docs} dokumen", "Jumlah total dokumen naskah yang berhasil diekstraksi."),
        ("Total Pasangan Dianalisis", f"{total_pairs:,} pasangan", "Jumlah seluruh kombinasi perbandingan berpasangan n*(n-1)/2."),
        ("Dokumen Memenuhi Syarat (PASS)", f"{passed_docs} dokumen ({pass_rate:.1f}%)", "Dokumen dengan skor tertinggi <= batas aman kelulusan."),
        ("Dokumen Melebihi Batas (FAIL)", f"{failed_docs} dokumen ({100 - pass_rate:.1f}%)", "Dokumen yang memerlukan revisi atau pemeriksaan komprehensif."),
        ("Waktu Pembuatan Laporan", time.strftime("%Y-%m-%d %H:%M:%S"), "Waktu komputasi dan penulisan file laporan Excel selesai.")
    ]

    for m_param, m_val, m_desc in metadata_rows:
        r_num = ws_meta.max_row + 1
        ws_meta.append([m_param, m_val, m_desc])
        row_cells = [ws_meta.cell(row=r_num, column=c) for c in range(1, len(meta_headers) + 1)]
        row_cells[0].font = Font(name="Segoe UI", size=10, bold=True, color="1F4E79")
        row_cells[1].font = Font(name="Segoe UI", size=10, bold=True)
        row_cells[1].alignment = Alignment(horizontal="center")
        row_cells[2].font = Font(name="Segoe UI", size=9)
        for cell in row_cells:
            cell.border = BORDER_BOX
        ws_meta.row_dimensions[r_num].height = 22

    ws_meta.freeze_panes = "A2"
    auto_fit_columns(ws_meta, min_w=15, max_w=65)

    # Simpan file
    wb.save(output_file)
    return output_file


if __name__ == "__main__":
    import pandas as pd

    # Script testing / direct upgrade on existing file
    excel_input = "Turnitin_Similarity_Report_P3MD.xlsx"
    if os.path.exists(excel_input):
        print(f"📖 Membaca file contoh: {excel_input}...")
        wb_in = openpyxl.load_workbook(excel_input, data_only=True)
        ws_summary = wb_in["Similarity Summary"]
        data = list(ws_summary.iter_rows(values_only=True))
        df_in = pd.DataFrame(data[1:], columns=data[0])

        passages_in = []
        if "Matched Passages Detail" in wb_in.sheetnames:
            ws_p = wb_in["Matched Passages Detail"]
            p_data = list(ws_p.iter_rows(values_only=True))
            if len(p_data) > 1:
                p_headers = p_data[0]
                for r in p_data[1:]:
                    pair_str = str(r[0] or "")
                    score_str = str(r[1] or "0%")
                    passage_text = str(r[2] or "")
                    parts = re.split(r"\s*<-->\s*|\s+vs\s+", pair_str)
                    d1 = parts[0].strip() if len(parts) > 0 else "-"
                    d2 = parts[1].strip() if len(parts) > 1 else "-"
                    
                    try:
                        s_val = float(score_str.replace("%", "").strip())
                    except:
                        s_val = 0.0

                    status = "FAIL" if s_val > 15.0 else "PASS"
                    passages_in.append({
                        "doc1": d1,
                        "doc2": d2,
                        "score": s_val,
                        "status": status,
                        "text": passage_text
                    })

        print(f"⚙️ Memproses ulang {len(df_in)} baris pasangan & {len(passages_in)} kutipan kalimat...")
        out = generate_excel_report(
            df_results=df_in,
            total_docs=len(set(df_in["Dokumen 1"]).union(set(df_in["Dokumen 2"]))),
            min_words=6,
            commander_threshold=17.0,
            pass_threshold=15.0,
            matched_passages=passages_in,
            output_file="Turnitin_Similarity_Report_P3MD.xlsx"
        )
        print(f"✅ Selesai! File laporan baru tersimpan di: {out}")


In [ ]:
# @title 5. Antarmuka Web Interaktif (Gradio - 100% Gratis, 72 Jam Uptime)
import os
import time
import pandas as pd
import gradio as gr

PUBLIC_DRIVE_URL = "https://drive.google.com/drive/u/0/folders/1YbKgSou6XhmCr1CLD2dRWy_ahzQ3HFDi"

def load_latest_leaderboard(target_dir="./dokumen_tugas_p3md"):
    """
    Memuat data laporan terbaru saat aplikasi pertama kali dibuka.
    Mengecek direktori target, direktori saat ini, atau mengunduh baseline jika belum ada.
    """
    excel_candidates = [
        os.path.join(target_dir, "Turnitin_Similarity_Report_P3MD.xlsx"),
        "Turnitin_Similarity_Report_P3MD.xlsx"
    ]
    
    excel_path = None
    for p in excel_candidates:
        if os.path.exists(p):
            excel_path = p
            break

    # Jika file belum ada, coba unduh baseline dari GitHub raw
    if not excel_path:
        raw_url = "https://raw.githubusercontent.com/egxl/Turnitin_Similaritas_P3MD/main/Turnitin_Similarity_Report_P3MD.xlsx"
        try:
            import urllib.request
            target_p = os.path.join(target_dir, "Turnitin_Similarity_Report_P3MD.xlsx")
            os.makedirs(target_dir, exist_ok=True)
            urllib.request.urlretrieve(raw_url, target_p)
            if os.path.exists(target_p):
                excel_path = target_p
        except Exception:
            pass

    if excel_path and os.path.exists(excel_path):
        try:
            df = pd.read_excel(excel_path, sheet_name="👤 Rekap Per Peserta")
            failed = sum(1 for s in df["Status Kelulusan"] if str(s).upper() == "FAIL")
            passed = len(df) - failed
            total_pairs = (len(df) * (len(df) - 1)) // 2
            summary_md = f"""### 📊 Ringkasan Eksekutif Similaritas Cohort P3MD (Data Terbaru)
- **Total Dokumen Peserta:** {len(df)} file
- **Kelulusan Cohort:** ✅ **{passed} LULUS** ({passed/len(df)*100:.1f}%) | ❌ **{failed} MELEBIHI BATAS** ({failed/len(df)*100:.1f}%)
- **Total Pasangan Diuji:** {total_pairs:,} pasang
- ℹ️ *Data di bawah adalah hasil analisis tersimpan terbaru. Unggah file tugas baru ke Google Drive lalu klik tombol **"🚀 Mulai Analisis Similaritas / Cek Dokumen Baru"** untuk memperbarui data.*
"""
            return summary_md, df, excel_path
        except Exception:
            pass

    # Fallback kosong jika benar-benar belum ada data sama sekali
    empty_df = pd.DataFrame(columns=[
        "Rank", "Nama Dokumen (Peserta)", "Status Kelulusan", "Skor Tertinggi (%)", 
        "Kategori Turnitin", "Pasangan Paling Mirip (Top Match)", "Skor Match #1 (%)", 
        "Pasangan Match #2", "Skor Match #2 (%)", "Rata-rata Similaritas Cohort (%)", 
        "Jumlah Pasangan > Batas", "Total Kata"
    ])
    default_md = """### 📊 Ringkasan Eksekutif Similaritas Cohort P3MD
Belum ada data analisis tersimpan. Silakan unggah file tugas ke Google Drive lalu klik tombol **"🚀 Mulai Analisis Similaritas / Cek Dokumen Baru"** di atas.
"""
    return default_md, empty_df, None

def run_analysis_pipeline(drive_url, min_words, pass_thresh, drop_quotes, drop_bib, force_recompute, progress=gr.Progress()):
    target_dir = "./dokumen_tugas_p3md"
    os.makedirs(target_dir, exist_ok=True)
    db_path = os.path.join(target_dir, "similarity_cache.db")
    
    if force_recompute and os.path.exists(db_path):
        try:
            os.remove(db_path)
        except:
            pass

    progress(0.02, desc="Menyiapkan sistem analisis...")
    log_messages = []
    def log(msg):
        log_messages.append(msg)
        print(msg)

    # 1. Sinkronisasi Dokumen Google Drive (0.05 -> 0.25)
    active_drive_url = (drive_url or "").strip() or PUBLIC_DRIVE_URL
    progress(0.05, desc="Menghubungi Google Drive...")
    
    def drive_prog_cb(curr, total, desc_text):
        frac = 0.05 + 0.20 * (curr / max(total, 1))
        progress(frac, desc=f"Google Drive [{curr}/{total}]: {desc_text[:40]}")

    dl, sk = sync_drive_folder(active_drive_url, target_dir, log_callback=log, progress_callback=drive_prog_cb)
    
    # 2. Pindai Dokumen Lokal
    supported_exts = {".docx", ".pdf", ".txt"}
    file_paths = []
    for root, _, f_list in os.walk(target_dir):
        for f in f_list:
            if os.path.splitext(f)[1].lower() in supported_exts and not f.startswith("~"):
                file_paths.append(os.path.join(root, f))

    if len(file_paths) < 2:
        return (
            f"⚠️ Ditemukan {len(file_paths)} dokumen di folder tugas. Minimal diperlukan 2 dokumen untuk analisis.\n\n"
            f"Silakan unggah dokumen ke folder Google Drive terlebih dahulu:\n{active_drive_url}",
            pd.DataFrame(),
            None,
            pd.DataFrame(),
            ""
        )

    # 3. Sinkronisasi SQLite Dokumen & Ekstraksi Teks (0.25 -> 0.50)
    progress(0.25, desc="Mempersiapkan database cache SQLite...")
    cache = TurnitinDBCache(db_path=db_path)
    
    # Bersihkan dokumen cache yang file fisiknya sudah dihapus dari disk
    active_basenames = [os.path.basename(p) for p in file_paths]
    pruned_c = cache.cleanup_deleted_documents(active_basenames)
    if pruned_c > 0:
        log(f"🧹 Menghapus {pruned_c} dokumen usang dari database cache SQLite.")
    
    def doc_prog_cb(curr, total, name):
        frac = 0.25 + 0.25 * (curr / max(total, 1))
        progress(frac, desc=f"Ekstraksi teks [{curr}/{total}]: {name[:35]}")

    doc_db, loaded_c, new_c = cache.sync_documents(
        file_paths, drop_quotes=drop_quotes, drop_bib=drop_bib, progress_callback=doc_prog_cb
    )

    # 4. Hitung Inkremental Pasangan (0.50 -> 0.85)
    progress(0.50, desc="Kalkulasi similaritas inkremental Turnitin...")
    def pair_prog_cb(curr, total):
        frac = 0.50 + 0.35 * (curr / max(total, 1))
        progress(frac, desc=f"Menghitung pasangan baru [{curr}/{total}]...")

    total_pairs, cached_pairs, new_pairs = cache.run_incremental_comparisons(
        doc_db, k_val=int(min_words), threshold=float(pass_thresh), progress_callback=pair_prog_cb
    )

    # 5. Buat Laporan Excel Komprehensif (0.85 -> 0.95)
    progress(0.85, desc="Menyusun data rekapitulasi...")
    df_results = cache.get_results_dataframe(list(doc_db.keys()), k_val=int(min_words))
    
    # Kumpulkan matched passages untuk detail bukti teks
    all_passages = []
    for _, r in df_results.iterrows():
        if r.get("Matches", 0) > 0 and isinstance(r.get("Passages"), list):
            s_val = float(r.get("Turnitin Max Score (%)", 0.0))
            p_status = "FAIL" if s_val > float(pass_thresh) else "PASS"
            for p_text in r["Passages"]:
                all_passages.append({
                    "doc1": r["Dokumen 1"],
                    "doc2": r["Dokumen 2"],
                    "score": s_val,
                    "status": p_status,
                    "text": p_text
                })

    progress(0.90, desc="Menyusun workbook Excel 7-Sheet...")
    excel_path = os.path.join(target_dir, "Turnitin_Similarity_Report_P3MD.xlsx")
    generate_excel_report(
        df_results, len(doc_db), min_words=int(min_words), 
        commander_threshold=COMMANDER_THRESHOLD, pass_threshold=float(pass_thresh),
        matched_passages=all_passages, output_file=excel_path
    )

    # Hitung Leaderboard 1 Baris Per Peserta untuk Tampilan Web
    progress(0.95, desc="Menghasilkan Leaderboard per peserta...")
    leaderboard = compute_leaderboard(df_results, threshold=float(pass_thresh))
    for i, item in enumerate(leaderboard, 1):
        item["Rank"] = i
        item["Nama Dokumen (Peserta)"] = item["Dokumen"]
        item["Pasangan Paling Mirip (Top Match)"] = item["Top Matched Document"]
        item["Skor Match #1 (%)"] = item["Top Match Score (%)"]
        item["Pasangan Match #2"] = item["2nd Matched Document"]
        item["Skor Match #2 (%)"] = item["2nd Match Score (%)"]

    df_display = pd.DataFrame(leaderboard)[[
        "Rank", "Nama Dokumen (Peserta)", "Status Kelulusan", "Skor Tertinggi (%)", 
        "Kategori Turnitin", "Pasangan Paling Mirip (Top Match)", "Skor Match #1 (%)", 
        "Pasangan Match #2", "Skor Match #2 (%)", "Rata-rata Similaritas Cohort (%)", 
        "Jumlah Pasangan > Batas", "Total Kata"
    ]]

    # 6. Otomatis Unggah Cache Database dan Laporan Excel ke Google Drive (0.95 -> 0.99)
    folder_id = extract_folder_id(active_drive_url)
    drive_upload_success = False
    if folder_id:
        progress(0.97, desc="Mengunggah cache database ke Google Drive...")
        up_db = upload_file_to_drive(db_path, folder_id, log_callback=log)
        up_xl = upload_file_to_drive(excel_path, folder_id, log_callback=log)
        drive_upload_success = up_db or up_xl

    failed_docs_count = sum(1 for d in leaderboard if d["Status Kelulusan"] == "FAIL")
    passed_docs_count = len(leaderboard) - failed_docs_count
    fail_pairs_count = len(df_results[df_results["Turnitin Max Score (%)"] > float(pass_thresh)])

    sync_note = "☁️ **Cache database & Excel tersinkron ke Google Drive.**" if drive_upload_success else "💾 **Cache tersimpan secara lokal.**"

    summary_md = f"""### 📊 Ringkasan Eksekutif Similaritas Cohort P3MD
- **Total Dokumen Peserta:** {len(doc_db)} file (📦 Dari Cache: {loaded_c}, 🆕 Baru Diunduh/Diproses: {new_c})
- **Kelulusan Cohort:** ✅ **{passed_docs_count} LULUS** ({passed_docs_count/len(leaderboard)*100:.1f}%) | ❌ **{failed_docs_count} MELEBIHI BATAS** ({failed_docs_count/len(leaderboard)*100:.1f}%)
- **Total Pasangan Diuji:** {total_pairs:,} pasang (⚡ Dari Cache: {cached_pairs:,}, 🔍 Baru Dihitung: {new_pairs:,})
- **Pasangan Melanggar Batas ({pass_thresh}%):** {fail_pairs_count} pasang
- **Waktu Hemat:** ~{round((cached_pairs * 0.005) / 60, 1)} menit berkat SQLite Cache!
- {sync_note}
"""
    progress(1.0, desc="Selesai!")
    return summary_md, df_display, excel_path, df_display, ""

# Tampilan Web Gradio
init_summary, init_df, init_excel = load_latest_leaderboard()

with gr.Blocks(title="Turnitin Document Similarity - P3MD", theme=gr.themes.Soft(), css=".dataframe-table { font-size: 13.5px !important; }") as demo:
    gr.Markdown("""
    # 🔍 Turnitin Document Similarity Checker (P3MD)
    **Sistem Deteksi Similaritas Dokumen Tugas Cohort P3MD Berbasis Standar Turnitin Resmi**
    """)

    with gr.Group():
        gr.Markdown(f"""
        ### 📋 Alur Kerja Pengumpulan Dokumen & Pemeriksaan
        Ikuti 4 langkah mudah berikut untuk memeriksa dokumen tugas Anda:

        1. **Unggah File Tugas ke Google Drive:**  
           Klik tombol **"📂 Buka Folder Google Drive P3MD"** di bawah untuk membuka folder pengumpulan tugas. Masukkan naskah tugas Anda (format `.docx`, `.pdf`, atau `.txt`) langsung ke dalam folder tersebut.
        2. **Jalankan Analisis Similaritas:**  
           Setelah file berhasil diunggah ke Google Drive, klik tombol **"🚀 Mulai Analisis Similaritas / Cek Dokumen Baru"**. Sistem akan otomatis mendeteksi dan mengunduh file baru Anda tanpa mengulang unduhan file lama.
        3. **Pantau Kemajuan (*Progress Bar*):**  
           Bilah kemajuan di bagian atas akan menampilkan progres secara realtime mulai dari sinkronisasi Google Drive, ekstraksi teks dokumen, kalkulasi pasangan Turnitin, hingga auto-upload cache database.
        4. **Lihat Hasil & Unduh Laporan Excel:**  
           Gunakan fitur **Cari Dokumen** di bawah untuk menemukan nama dokumen Anda pada tabel **Rekap Per Peserta (*Leaderboard*)**, dan klik tombol unduh untuk mengunduh laporan resmi Excel 7-Sheet lengkap.
        """)

        with gr.Row():
            drive_link_btn = gr.Button(
                "📂 Buka Folder Google Drive P3MD (Upload Dokumen Di Sini) ↗",
                variant="secondary",
                size="lg",
                link=PUBLIC_DRIVE_URL
            )
        gr.Markdown(f"🔗 *Tautan Alternatif Folder Drive:* [{PUBLIC_DRIVE_URL}]({PUBLIC_DRIVE_URL})")

    with gr.Row():
        run_btn = gr.Button("🚀 Mulai Analisis Similaritas / Cek Dokumen Baru", variant="primary", size="lg")

    with gr.Accordion("⚙️ Parameter Analisis & Pengaturan Lanjutan (Opsional)", open=False):
        drive_input = gr.Textbox(
            label="📁 Link Folder Google Drive (Default: Folder Publik Tugas P3MD)",
            value=PUBLIC_DRIVE_URL,
            placeholder="https://drive.google.com/drive/folders/..."
        )
        with gr.Row():
            min_words_slider = gr.Slider(minimum=4, maximum=12, value=6, step=1, label="Min Consecutive Words (Standar Turnitin: 6)")
            thresh_slider = gr.Slider(minimum=5.0, maximum=50.0, value=15.0, step=1.0, label="Batas Toleransi Kelulusan (%)")
        with gr.Row():
            quotes_cb = gr.Checkbox(value=True, label="Abaikan Kutipan (\" \")")
            bib_cb = gr.Checkbox(value=True, label="Abaikan Daftar Pustaka")
            force_cb = gr.Checkbox(value=False, label="Paksa Hitung Ulang Semua (Reset Cache)")

    status_output = gr.Markdown(value=init_summary)
    with gr.Row():
        download_btn = gr.File(
            value=init_excel,
            label="📥 Unduh Laporan Excel Resmi (Turnitin_Similarity_Report_P3MD.xlsx)"
        )
    
    # Search Bar & Filter Controls
    with gr.Row():
        search_input = gr.Textbox(
            label="🔍 Cari Dokumen / Nama Peserta",
            placeholder="Ketik nama file atau peserta untuk menyaring tabel (contoh: Fauzi, Tariq, Rayga, atau status: PASS/FAIL)...",
            scale=5
        )
        reset_search_btn = gr.Button("🔄 Reset Pencarian", scale=1, variant="secondary")

    # Full Page Recap Table
    table_output = gr.Dataframe(
        value=init_df,
        label="👤 Rekap Hasil Per Peserta (Leaderboard 1 Baris Per Dokumen - Diurutkan dari Skor Tertinggi)",
        interactive=False,
        wrap=True,
        height=750
    )

    current_df_state = gr.State(value=init_df)

    def filter_table(query, full_df):
        if full_df is None or len(full_df) == 0:
            return full_df
        if not query or not query.strip():
            return full_df
        q = query.strip().lower()
        mask = full_df.astype(str).apply(lambda row: row.str.lower().str.contains(q, regex=False).any(), axis=1)
        return full_df[mask]

    def reset_search(full_df):
        return "", full_df

    search_input.change(fn=filter_table, inputs=[search_input, current_df_state], outputs=[table_output])
    reset_search_btn.click(fn=reset_search, inputs=[current_df_state], outputs=[search_input, table_output])

    run_btn.click(
        fn=run_analysis_pipeline,
        inputs=[drive_input, min_words_slider, thresh_slider, quotes_cb, bib_cb, force_cb],
        outputs=[status_output, table_output, download_btn, current_df_state, search_input]
    )

# Luncurkan web app publik (72 jam gratis, tanpa kartu kredit)
demo.queue().launch(share=True, debug=False)


In [ ]:
# @title 6. Alternatif: Eksekusi Langsung Tanpa Web UI (Batch Mode)
# Jalankan sel ini HANYA jika ingin langsung mengeksekusi di Colab tanpa membuka Web UI
jalankan_batch_mode = False  # @param {type:"boolean"}

if not jalankan_batch_mode:
    print("ℹ️ Batch Mode dilewati secara otomatis (Web UI pada Sel 5 sudah aktif).")
    print("👉 Silakan gulir ke atas ke Sel 5 untuk menggunakan Web UI atau membuka link gradio.live.")
    print("👉 Jika ingin menjalankan Batch Mode langsung tanpa Web UI, centang 'jalankan_batch_mode' di atas lalu jalankan kembali sel ini.")
else:
    direct_drive_url = "https://drive.google.com/drive/u/0/folders/1YbKgSou6XhmCr1CLD2dRWy_ahzQ3HFDi"  # @param {type:"string"}
    folder_dir = "./dokumen_tugas_p3md"

    if direct_drive_url.strip():
        sync_drive_folder(direct_drive_url, folder_dir)

    file_paths = [
        os.path.join(root, f) 
        for root, _, files in os.walk(folder_dir) 
        for f in files 
        if os.path.splitext(f)[1].lower() in {".docx", ".pdf", ".txt"} and not f.startswith("~")
    ]

    if len(file_paths) >= 2:
        cache = TurnitinDBCache(os.path.join(folder_dir, "similarity_cache.db"))
        doc_db, loaded_c, new_c = cache.sync_documents(file_paths, drop_quotes=True, drop_bib=True)
        tot, cached_p, new_p = cache.run_incremental_comparisons(doc_db, k_val=6, threshold=PASS_THRESHOLD)
        df_res = cache.get_results_dataframe(list(doc_db.keys()), k_val=6)
        
        # Kumpulkan matched passages untuk laporan detail
        all_passages = []
        for _, r in df_res.iterrows():
            if r.get("Matches", 0) > 0 and isinstance(r.get("Passages"), list):
                s_val = float(r.get("Turnitin Max Score (%)", 0.0))
                p_status = "FAIL" if s_val > PASS_THRESHOLD else "PASS"
                for p_text in r["Passages"]:
                    all_passages.append({
                        "doc1": r["Dokumen 1"], "doc2": r["Dokumen 2"],
                        "score": s_val, "status": p_status, "text": p_text
                    })

        out_file = generate_excel_report(
            df_res, len(doc_db), min_words=6,
            commander_threshold=COMMANDER_THRESHOLD, pass_threshold=PASS_THRESHOLD,
            matched_passages=all_passages, output_file="Turnitin_Similarity_Report_P3MD.xlsx"
        )
        print(f"🎉 Selesai! Laporan tersimpan di: {out_file}")
        try:
            from google.colab import files
            files.download(out_file)
        except:
            pass
    else:
        print(f"⚠️ Minimal 2 dokumen diperlukan, ditemukan {len(file_paths)}.")
